# 12.6 · T5 与编码器-解码器 / T5 & Encoder-Decoder Models

> **课程定位 / Where this fits**
> 第 6 课，**Part 12**。Transformer 的第三种用法：**编码器 + 解码器**一起用。
> Lesson 6, **Part 12**. The third Transformer use: **encoder + decoder** together.
>
> BERT(编码器)擅理解、GPT(解码器)擅生成。**编码-解码**架构两者兼得：编码器**双向理解输入**，解码器**自回归生成输出**，中间用**交叉注意力(cross-attention)** 把两者连起来——天生适合"输入→输出"的**转换(transduction)** 任务(翻译、摘要、改写)。**T5**(2019)更进一步：把**所有 NLP 任务统一成"文本到文本"**(text-to-text)——分类、问答、翻译、摘要全都是"输入一段文本、输出一段文本"，用同一个模型同一套接口。本课**从零实现带交叉注意力的编码-解码 Transformer**。
> BERT (encoder) excels at understanding, GPT (decoder) at generation. The **encoder-decoder** gets both: the encoder **understands the input bidirectionally**, the decoder **generates autoregressively**, joined by **cross-attention** — naturally suited to "input→output" **transduction** (translation, summarization, rewriting). **T5** (2019) goes further: **unify every NLP task as "text-to-text"** — classification, QA, translation, summarization are all "input text → output text" with one model and interface. We **implement an encoder-decoder Transformer with cross-attention from scratch**.
>
> 💼 **实战/面试视角**："交叉注意力 / 编码-解码 vs 纯解码器 / T5 的 text-to-text 框架 / span corruption" 常考。
> 💼 **Practical/interview angle:** "cross-attention / encoder-decoder vs decoder-only / T5's text-to-text / span corruption" — common.

> 📐 **符号约定 / Notation**
> - 自注意力 —— Q,K,V 都来自同一序列 / self-attention: Q,K,V from the same sequence
> - 交叉注意力 —— Q 来自解码器, K,V 来自编码器 / cross-attention: Q from decoder, K,V from encoder

> 💡 **面试相关 / Interview-relevant**
> - "交叉注意力和自注意力的区别"（出镜率 ★★★★★）
> - "编码-解码 vs 纯解码器(T5 vs GPT)各适合什么"（★★★★）
> - "T5 的 text-to-text 统一框架"（★★★★）
> - "T5 的 span corruption 预训练"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解编码-解码架构与三种 Transformer 用法的区别。
   Understand the encoder-decoder and the three Transformer uses.
2. **从零实现交叉注意力**，懂它和自注意力的区别。
   Implement cross-attention from scratch; distinguish it from self-attention.
3. 搭一个编码-解码 Transformer 做"文本到文本"任务。
   Build an encoder-decoder Transformer for a text-to-text task.
4. 理解 T5 的统一框架与 span corruption。
   Understand T5's unified framework and span corruption.

## 目录 / TOC
1. [三种架构与 text-to-text ⭐](#1)
2. [交叉注意力（从零）⭐](#2)
3. [编码-解码 Transformer：训练 ⭐](#3)
4. [T5 统一框架 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 三种架构与 text-to-text ⭐ / Three Architectures & Text-to-Text

Transformer 的三种用法(面试常让你对比)：
The three Transformer uses (interviewers love the comparison):

| 架构 | 代表 | 注意力 | 擅长 |
|---|---|---|---|
| **编码器** Encoder | BERT | 双向 | 理解(分类/抽取/NER) |
| **解码器** Decoder | GPT | 因果单向 | 生成(续写/对话) |
| **编码-解码** Enc-Dec | T5 / 原始Transformer | 编码器双向 + 解码器因果 + 交叉注意力 | 转换(翻译/摘要) |

**T5 的大一统思想(text-to-text)**：与其为每个任务设计不同的输出层，不如**把所有任务都变成"输入文本→输出文本"**：
**T5's unifying idea (text-to-text):** instead of task-specific output heads, **cast every task as "input text → output text":**
- 翻译：`"translate English to German: That is good." → "Das ist gut."`
  Translation: `"translate English to German: ..." → "..."`
- 分类：`"sentiment: I love this movie" → "positive"`
  Classification: `"sentiment: I love this movie" → "positive"`
- 摘要：`"summarize: <长文>" → "<摘要>"`
  Summarization: `"summarize: <long text>" → "<summary>"`

好处：**一个模型、一套损失(都是生成文本)、一个接口**处理所有任务。这种"任务即文本指令"的思路也直接启发了后来的**指令微调和 prompt**(12.11)。
Benefit: **one model, one loss (text generation), one interface** for all tasks. This "task as a text instruction" idea directly inspired **instruction tuning and prompting** (12.11).


<a id="2"></a>
## 2. 交叉注意力（从零）⭐ / Cross-Attention From Scratch

编码-解码相比纯解码器，多了一个关键组件：**交叉注意力(cross-attention)**。回忆自注意力：Q、K、V 都来自**同一个序列**。交叉注意力不同：
The encoder-decoder adds one key component over decoder-only: **cross-attention**. Recall self-attention: Q, K, V all come from the **same sequence**. Cross-attention differs:
- **Query 来自解码器**(当前正在生成的位置)。
  **Query from the decoder** (the position being generated).
- **Key、Value 来自编码器输出**(整个输入的表示)。
  **Key, Value from the encoder output** (the input's representation).

直觉：解码器在生成每个输出词时，用交叉注意力**"回看"整个输入**，决定该关注输入的哪些部分——这正是 12.2 注意力机制的"翻译对齐"，现在长在 Transformer 里。
Intuition: when generating each output word, the decoder uses cross-attention to **"look back" at the entire input**, deciding which input parts to focus on — exactly 12.2's "translation alignment," now inside the Transformer.

所以**解码器块有两种注意力**：①**带因果掩码的自注意力**(看已生成的输出)；②**交叉注意力**(看编码器的输入表示)。
So a **decoder block has two attentions**: ① **masked self-attention** (over generated output); ② **cross-attention** (over the encoder's input representation).


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, math, time
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid")
torch.manual_seed(0); np.random.seed(0)

def attention(Q, K, V, mask=None):
    d = Q.size(-1); sc = Q @ K.transpose(-2, -1) / math.sqrt(d)
    if mask is not None: sc = sc.masked_fill(mask == 0, float("-inf"))
    return F.softmax(sc, -1) @ V

class MHA(nn.Module):
    """支持自注意力和交叉注意力 / supports both self- and cross-attention."""
    def __init__(s, D, H):
        super().__init__(); s.H = H; s.d = D//H
        s.wq = nn.Linear(D, D); s.wk = nn.Linear(D, D); s.wv = nn.Linear(D, D); s.o = nn.Linear(D, D)
    def split(s, x): B, T, D = x.shape; return x.reshape(B, T, s.H, s.d).transpose(1, 2)
    def forward(s, q_in, kv_in, mask=None):                # q_in→Q; kv_in→K,V (自注意力时二者相同) / same for self-attn
        B, T, D = q_in.shape
        Q, K, Vv = s.split(s.wq(q_in)), s.split(s.wk(kv_in)), s.split(s.wv(kv_in))
        o = attention(Q, K, Vv, mask).transpose(1, 2).reshape(B, T, D)
        return s.o(o)
print("MHA: q_in 提供 Query; kv_in 提供 Key/Value")
print("  自注意力: q_in = kv_in (同一序列)")
print("  交叉注意力: q_in = 解码器状态, kv_in = 编码器输出 (解码器'回看'输入)")


<a id="3"></a>
## 3. 编码-解码 Transformer：训练 ⭐ / Encoder-Decoder: Training

组装完整的编码-解码 Transformer：
Assemble a full encoder-decoder Transformer:
- **编码器块**：双向自注意力 + FFN(理解输入)。
  **Encoder block:** bidirectional self-attention + FFN (understand input).
- **解码器块**：带因果掩码的自注意力 + **交叉注意力**(看编码器输出) + FFN。
  **Decoder block:** masked self-attention + **cross-attention** (over encoder output) + FFN.

我们用一个"文本到文本"玩具任务：**把一串数字排序**(输入 `[7,3,9,1]` → 输出 `[1,3,7,9]`)。排序需要解码器**全局回看输入**(交叉注意力)，很能体现编码-解码的价值。
We use a text-to-text toy task: **sort a list of numbers** (input `[7,3,9,1]` → output `[1,3,7,9]`). Sorting needs the decoder to **globally look back at the input** (cross-attention), showcasing the encoder-decoder.


In [ ]:
V = 16; PAD, BOS, EOS = 0, 1, 2; DIG = list(range(3, 13))
def gen(n, L):
    src = [list(np.random.choice(DIG, L)) for _ in range(n)]
    return src, [sorted(s) for s in src]                  # 目标 = 排序后的输入 / target = sorted input

class EncBlock(nn.Module):
    def __init__(s, D, H): super().__init__(); s.a=MHA(D,H); s.n1=nn.LayerNorm(D); s.n2=nn.LayerNorm(D); s.ff=nn.Sequential(nn.Linear(D,4*D),nn.ReLU(),nn.Linear(4*D,D))
    def forward(s, x): h=s.n1(x); x=x+s.a(h,h); return x+s.ff(s.n2(x))   # 双向自注意力 / bidirectional self-attn
class DecBlock(nn.Module):
    def __init__(s, D, H):
        super().__init__(); s.sa=MHA(D,H); s.ca=MHA(D,H)
        s.n1=nn.LayerNorm(D); s.n2=nn.LayerNorm(D); s.n3=nn.LayerNorm(D); s.ff=nn.Sequential(nn.Linear(D,4*D),nn.ReLU(),nn.Linear(4*D,D))
    def forward(s, x, enc, cmask):
        h=s.n1(x); x=x+s.sa(h, h, cmask)                  # ① 带因果掩码的自注意力 / masked self-attn
        h=s.n2(x); x=x+s.ca(h, enc)                       # ② 交叉注意力: Q=解码器, K/V=编码器 / cross-attn
        return x+s.ff(s.n3(x))
class EncoderDecoder(nn.Module):
    def __init__(s, V, D=64, H=4, L=2, maxlen=20):
        super().__init__(); s.emb=nn.Embedding(V,D); s.pe=nn.Parameter(torch.randn(1,maxlen,D)*0.02)
        s.enc=nn.ModuleList([EncBlock(D,H) for _ in range(L)]); s.dec=nn.ModuleList([DecBlock(D,H) for _ in range(L)]); s.head=nn.Linear(D,V)
    def forward(s, src, tgt):
        e = s.emb(src)+s.pe[:, :src.size(1)]
        for b in s.enc: e = b(e)                           # 编码器: 理解输入 / encode input
        d = s.emb(tgt)+s.pe[:, :tgt.size(1)]; T=tgt.size(1); cmask=torch.tril(torch.ones(T,T)).view(1,1,T,T)
        for b in s.dec: d = b(d, e, cmask)                 # 解码器: 自注意力+交叉注意力 / decode
        return s.head(d)

src, tgt = gen(4000, 8)
S = torch.tensor(src); Tin = torch.tensor([[BOS]+t for t in tgt]); Tout = torch.tensor([t+[EOS] for t in tgt])
torch.manual_seed(0); m = EncoderDecoder(V); opt = torch.optim.Adam(m.parameters(), 3e-3); ce = nn.CrossEntropyLoss()
t0 = time.time()
for ep in range(15):
    for i in range(0, len(S), 128):
        lg = m(S[i:i+128], Tin[i:i+128]); opt.zero_grad(); ce(lg.reshape(-1,V), Tout[i:i+128].reshape(-1)).backward(); opt.step()
sv, tv = gen(500, 8); Sv=torch.tensor(sv); Tinv=torch.tensor([[BOS]+t for t in tv]); Toutv=torch.tensor([t+[EOS] for t in tv])
acc = (m(Sv, Tinv).argmax(2) == Toutv).all(1).float().mean().item()
print(f"编码-解码 Transformer 排序任务整句准确率 = {acc:.3f} ({time.time()-t0:.0f}s)")
print(f"示例: 输入 {[int(x) for x in sv[0]]} → 目标 {[int(x) for x in tv[0]]}")
print("解码器靠'带因果掩码的自注意力'(看已生成) + '交叉注意力'(回看输入) 完成排序")


<a id="4"></a>
## 4. T5 统一框架 + 小结 ⭐ / T5 Unified Framework & Summary

**T5(Text-to-Text Transfer Transformer)** 把编码-解码架构 + text-to-text 思想发挥到极致：
**T5** pushes the encoder-decoder + text-to-text idea to the extreme:
- **预训练任务: span corruption(片段遮盖)**。不像 BERT 遮单个词，T5 **遮掉连续的一段词**，让解码器**生成被遮的片段**(输入留个哨兵标记, 输出生成被遮内容)。这天然契合"输入文本→输出文本"。
  **Pretraining: span corruption.** Unlike BERT masking single tokens, T5 **masks contiguous spans** and has the decoder **generate the missing spans** (input keeps a sentinel, output produces the masked content). Naturally text-to-text.
- **微调**：所有下游任务都加一个**任务前缀**(如 `"summarize:"`)，统一成生成。
  **Fine-tuning:** every task gets a **task prefix** (e.g. `"summarize:"`), unified as generation.

**编码-解码 vs 纯解码器(面试)**：编码-解码(T5/BART)对**输入输出界限清晰**的转换任务(翻译/摘要)更自然、参数利用更高效；纯解码器(GPT)更简单、更易 scale，靠把输入也当 prompt 拼进去也能做转换——**当今超大模型(GPT-4 等)多是纯解码器**，因为简单+可扩展性压倒一切。
**Encoder-decoder vs decoder-only (interview):** encoder-decoder (T5/BART) is natural and parameter-efficient for transduction with **clear input/output boundaries** (translation/summarization); decoder-only (GPT) is simpler and easier to scale, handling transduction by concatenating input as a prompt — **today's largest models (GPT-4 etc.) are mostly decoder-only**, since simplicity + scalability win.

```
三架构: 编码器(BERT,双向,理解) / 解码器(GPT,因果,生成) / 编码-解码(T5,转换)
交叉注意力: Q来自解码器, K/V来自编码器 → 解码器"回看"输入(对比自注意力Q/K/V同源)
解码器块 = 带因果掩码的自注意力 + 交叉注意力 + FFN
T5 text-to-text: 所有任务统一成"输入文本→输出文本"(加任务前缀); 一个模型一套损失
T5 预训练 = span corruption: 遮连续片段, 解码器生成被遮内容
编码-解码 vs 纯解码器: 前者转换任务自然/高效; 后者简单易scale → 当今超大模型多是纯解码器
```

### 💡 面试速查 / Interview cheat-sheet
1. **交叉注意力**: Q=解码器, K/V=编码器; 解码器回看输入(vs 自注意力同源)。
   Cross-attention: Q=decoder, K/V=encoder; decoder looks back at input.
2. **解码器块**: 因果自注意力 + 交叉注意力 + FFN(三件套)。
   Decoder block: causal self-attention + cross-attention + FFN.
3. **T5 text-to-text**: 所有任务=文本→文本, 加任务前缀, 一套接口。
   T5 text-to-text: every task = text→text with a task prefix, one interface.
4. **span corruption**: 遮连续片段让解码器生成(vs BERT 遮单词)。
   Span corruption: mask spans, decoder generates them (vs BERT single tokens).
5. **enc-dec vs decoder-only**: 转换任务前者自然; 后者更简单易scale(主流超大模型)。
   Enc-dec vs decoder-only: former natural for transduction; latter simpler/scalable (dominant at scale).

### 下一节 / Next
**12.7 分词**——前面 GPT/BERT 都假设有"token"。但文本怎么切成 token？字符级太长、词级词表爆炸且处理不了新词。现代大模型用**子词分词(BPE)**：在字符和词之间取平衡。我们会**从零实现 BPE 算法**。
**12.7 Tokenization** — GPT/BERT assumed "tokens." But how is text split into tokens? Char-level is too long, word-level explodes and fails on new words. Modern LLMs use **subword tokenization (BPE)**, balancing char and word. We'll **implement the BPE algorithm from scratch**.
